# Interactive Local Orthogonal Coordinate Transform (OCT) Explorer

This notebook allows you to interactively explore how the parameters of a LocalOCT affect the resulting coordinate grid.

## Parameters and Constraints

A LocalOCT in 2D is parameterized by:
- **p**: Base point (we fix at origin)
- **U**: Orthonormal frame - parameterized by rotation angle θ
- **log_s**: Log of loadings (s₁, s₂) - 2 free parameters
- **β**: Rotation coefficients - **symmetric** matrix, so only 3 free params (β₁₁, β₁₂=β₂₁, β₂₂)
- **dβ**: Computed from β via Lamé equations (flatness constraints)

### Geometric Interpretation
- **θ**: Rotation of the principal directions
- **log_s**: How stretched/compressed each coordinate direction is
- **β_ii**: Score coefficients - affect log-likelihood gradients
- **β_ij (i≠j)**: Rotation coefficients - control how coordinate curves bend


In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive, FloatText, FloatSlider, VBox, HBox, Label, Output, Button
import ipywidgets as widgets
from IPython.display import display, clear_output

# Add the local_coordinates package to path
import sys
sys.path.insert(0, '..')

from experimental.oct import LocalOCT, plot_oct_grid, _compute_dbeta_from_beta


In [2]:
def create_orthonormal_U(theta: float) -> jnp.ndarray:
    """Create a 2x2 orthonormal matrix from rotation angle theta."""
    c, s = jnp.cos(theta), jnp.sin(theta)
    return jnp.array([[c, -s], [s, c]])

def create_beta(beta_11: float, beta_12: float, beta_21: float, beta_22: float, enforce_symmetry: bool = True) -> jnp.ndarray:
    """Create a 2x2 beta matrix, optionally enforcing symmetry."""
    if enforce_symmetry:
        # Use beta_12 for both off-diagonal entries
        return jnp.array([[beta_11, beta_12], [beta_12, beta_22]])
    else:
        # Allow asymmetric beta
        return jnp.array([[beta_11, beta_12], [beta_21, beta_22]])

def create_oct_from_params(theta, log_s1, log_s2, beta_11, beta_12, beta_21, beta_22, enforce_symmetry=True):
    """Create a LocalOCT from the UI parameters."""
    p = jnp.zeros(2)
    U = create_orthonormal_U(theta)
    log_s = jnp.array([log_s1, log_s2])
    beta = create_beta(beta_11, beta_12, beta_21, beta_22, enforce_symmetry)
    dbeta = _compute_dbeta_from_beta(beta)

    return LocalOCT(p=p, U=U, log_s=log_s, beta=beta, dbeta=dbeta)


In [ ]:
# Create the output area for the plot
output = Output()

# Helper function to create a slider with linked text input
def make_slider_with_text(slider_class, text_class, value, min_val, max_val, step, description):
    """Create a slider + text input combo that are bidirectionally linked."""
    style = {'description_width': '100px'}
    slider = slider_class(
        value=value, min=min_val, max=max_val, step=step, description=description,
        style=style, layout=widgets.Layout(width='280px'), continuous_update=True, readout=False
    )
    text = text_class(value=value, step=step, layout=widgets.Layout(width='80px'))
    widgets.link((slider, 'value'), (text, 'value'))
    container = HBox([slider, text], layout=widgets.Layout())
    return slider, text, container

# Widget style
style = {'description_width': '120px'}
slider_width = '350px'

# Principal basis angle
theta_widget, theta_text, theta_box = make_slider_with_text(
    FloatSlider, widgets.FloatText, 0.0, -float(jnp.pi), float(jnp.pi), 0.05, 'θ (rad):'
)

# Log loadings
log_s1_widget, log_s1_text, log_s1_box = make_slider_with_text(
    FloatSlider, widgets.FloatText, 0.0, -10.0, 2.0, 0.05, 'log(s1):'
)

log_s2_widget, log_s2_text, log_s2_box = make_slider_with_text(
    FloatSlider, widgets.FloatText, 0.0, -10.0, 2.0, 0.05, 'log(s2):'
)

# Beta matrix widgets
beta_11_widget, beta_11_text, beta_11_box = make_slider_with_text(
    FloatSlider, widgets.FloatText, 0.0, -7.0, 7.0, 0.1, 'β11:'
)

beta_12_widget, beta_12_text, beta_12_box = make_slider_with_text(
    FloatSlider, widgets.FloatText, 0.0, -15.0, 15.0, 0.1, 'β12:'
)

# Separate beta_21 widget (only used when symmetry is not enforced)
beta_21_widget, beta_21_text, beta_21_box = make_slider_with_text(
    FloatSlider, widgets.FloatText, 0.0, -15.0, 15.0, 0.1, 'β21:'
)

beta_22_widget, beta_22_text, beta_22_box = make_slider_with_text(
    FloatSlider, widgets.FloatText, 0.0, -15.0, 15.0, 0.1, 'β22:'
)

# Symmetry enforcement toggle
symmetry_toggle = widgets.Checkbox(
    value=False,
    description='Enforce β symmetry (β12 = β21)',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

# Grid parameters
span_widget, span_text, span_box = make_slider_with_text(
    FloatSlider, widgets.FloatText, 0.3, 0.1, 1.0, 0.05, 'Grid span:'
)

# Toggle for showing coordinate grid
show_grid_toggle = widgets.Checkbox(
    value=True,
    description='Show coordinate grid',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

# Toggle for showing arrows (basis vectors)
show_arrows_toggle = widgets.Checkbox(
    value=False,
    description='Show basis vectors',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

# Dropdown for sample distribution selection
sample_distribution_widget = widgets.Dropdown(
    options=[('None', 'none'), ('Gaussian', 'gaussian'), ("Student's t", 'student_t')],
    value='none',
    description='Samples:',
    style=style,
    layout=widgets.Layout(width='250px')
)

# Gaussian standard deviation slider
sample_std_widget, sample_std_text, sample_std_box = make_slider_with_text(
    FloatSlider, widgets.FloatText, 0.1, 0.01, 0.5, 0.01, 'Sample σ:'
)

# Number of samples slider
n_samples_widget, n_samples_text, n_samples_box = make_slider_with_text(
    widgets.IntSlider, widgets.IntText, 500, 100, 2000, 100, 'N samples:'
)

# Degrees of freedom slider for Student's t
student_t_df_widget, student_t_df_text, student_t_df_box = make_slider_with_text(
    FloatSlider, widgets.FloatText, 3.0, 1.0, 30.0, 0.5, 'df (ν):'
)

# Update button (optional manual refresh)
update_button = Button(description='Update Plot', button_style='primary')

# Flag to prevent recursive updates
_updating = [False]

# Set initial visibility directly (before widgets are in a container)
# beta_21 starts hidden (symmetry enforced by default)
beta_21_box.layout.display = 'none'
# Sample controls start hidden (shown based on dropdown selection)
sample_std_box.layout.display = 'none'
n_samples_box.layout.display = 'none'
student_t_df_box.layout.display = 'none'


In [4]:
from jax import random
from local_coordinates.jet import Jet

def map_z_to_x(oct, z_samples):
    """Map points from z-space to x-space using the OCT's Taylor expansion."""
    jacobian = oct.get_jacobian()
    jet = Jet(
        value=oct.p,
        gradient=jacobian.value,
        hessian=jacobian.gradient,
    )
    # Evaluate the jet at each sample point
    x_samples = jax.vmap(lambda z: jet(z))(z_samples)
    return x_samples

def update_plot(b=None):
    """Update the plot with current widget values."""
    # Skip if we're in the middle of a programmatic update
    if _updating[0]:
        return

    # Update widget visibility based on toggle states
    # beta_21 visibility (use box container for slider+text combo)
    beta_21_box.layout.display = 'none' if symmetry_toggle.value else ''
    # Sample controls visibility based on dropdown
    sample_dist = sample_distribution_widget.value
    if sample_dist == 'none':
        sample_std_box.layout.display = 'none'
        n_samples_box.layout.display = 'none'
        student_t_df_box.layout.display = 'none'
    elif sample_dist == 'gaussian':
        sample_std_box.layout.display = ''
        n_samples_box.layout.display = ''
        student_t_df_box.layout.display = 'none'
    else:  # student_t
        sample_std_box.layout.display = ''
        n_samples_box.layout.display = ''
        student_t_df_box.layout.display = ''

    # Sync beta_21 to beta_12 when symmetry is enforced
    if symmetry_toggle.value and beta_21_widget.value != beta_12_widget.value:
        _updating[0] = True
        beta_21_widget.value = beta_12_widget.value
        _updating[0] = False

    with output:
        clear_output(wait=True)

        # Get current values
        theta = theta_widget.value
        log_s1 = log_s1_widget.value
        log_s2 = log_s2_widget.value
        beta_11 = beta_11_widget.value
        beta_12 = beta_12_widget.value
        beta_21 = beta_21_widget.value
        beta_22 = beta_22_widget.value
        span = span_widget.value
        enforce_symmetry = symmetry_toggle.value
        show_grid = show_grid_toggle.value
        show_arrows = show_arrows_toggle.value
        sample_dist = sample_distribution_widget.value
        sample_std = sample_std_widget.value
        n_samples = n_samples_widget.value
        student_t_df = student_t_df_widget.value

        # Create the OCT
        try:
            oct = create_oct_from_params(
                theta, log_s1, log_s2,
                beta_11, beta_12, beta_21, beta_22,
                enforce_symmetry=enforce_symmetry
            )

            # Create figure
            fig, ax = plt.subplots(figsize=(8, 8))

            # Build title showing mode
            sym_str = "symmetric β" if enforce_symmetry else "non-symmetric β"
            parts = []
            if show_grid:
                parts.append("Grid")
            if show_arrows:
                parts.append("Arrows")
            if sample_dist == 'gaussian':
                parts.append(f"Samples N(0, {sample_std:.2f}²)")
            elif sample_dist == 'student_t':
                parts.append(f"Samples t(ν={student_t_df:.1f}, σ={sample_std:.2f})")
            if parts:
                title = f'{" + ".join(parts)} ({sym_str})'
            else:
                title = f'Empty plot ({sym_str})'

            # Plot the coordinate grid
            plot_oct_grid(
                oct,
                num=21,
                span=span,
                show=False,
                ax=ax,
                title=title,
                draw_grid=show_grid,
                draw_basis_vectors=show_arrows
            )

            # If showing samples, draw them on top
            if sample_dist != 'none':
                # Generate samples in z-space
                key = random.PRNGKey(42)  # Fixed seed for reproducibility
                if sample_dist == 'student_t':
                    # Student's t distribution: scale by std dev
                    z_samples = random.t(key, df=student_t_df, shape=(n_samples, 2)) * sample_std
                else:
                    # Gaussian distribution
                    z_samples = random.normal(key, shape=(n_samples, 2)) * sample_std

                # Map to x-space
                x_samples = map_z_to_x(oct, z_samples)

                # Plot samples
                ax.scatter(
                    x_samples[:, 0], x_samples[:, 1],
                    c='#F4A024',  # Orange/gold color
                    s=8,
                    alpha=0.7,
                    edgecolors='#1a1a2e',
                    linewidths=0.3,
                    zorder=20
                )

            plt.tight_layout()
            plt.show()

        except Exception as e:
            import traceback
            traceback.print_exc()

# Connect button to update function
update_button.on_click(update_plot)

# Also update on value change for each widget
for w in [theta_widget, log_s1_widget, log_s2_widget,
          beta_11_widget, beta_12_widget, beta_21_widget, beta_22_widget,
          span_widget, symmetry_toggle, show_grid_toggle, show_arrows_toggle,
          sample_distribution_widget, sample_std_widget, n_samples_widget,
          student_t_df_widget]:
    w.observe(lambda change: update_plot(), names='value')


In [5]:
# Layout the widgets - vertical stack for sliders (use box containers for slider+text combos)
controls = VBox([
    Label('━━━ Principal Basis (Orthonormal Frame) ━━━'),
    theta_box,
    Label('━━━ Log Loadings ━━━'),
    log_s1_box,
    log_s2_box,
    Label('━━━ Rotation Coefficients β ━━━'),
    symmetry_toggle,
    beta_11_box,
    beta_12_box,
    beta_21_box,  # Hidden by default when symmetry is enforced
    beta_22_box,
    Label('━━━ Grid Settings ━━━'),
    span_box,
    Label('━━━ Visualization ━━━'),
    show_grid_toggle,
    show_arrows_toggle,
    sample_distribution_widget,
    sample_std_box,       # Hidden by default, shown when samples enabled
    n_samples_box,        # Hidden by default, shown when samples enabled
    student_t_df_box,     # Hidden by default, shown when Student's t selected
    update_button
])

# Display everything
display(HBox([controls, output]))

# Initial plot
update_plot()
